transducer settings for a moving standing wave
drive settings for two
opposing transducers on the jet axis:

 upstream transducer -> interaction point -> downstream transducer


Two opposing waves of slightly different frequency make a standing wave whose nodes drift toward the slower one:

* node spacing     Δ = c / (f_up + f_down)
* node (wavefront) velocity    v = c (f_up − f_down) / (f_up + f_down)

so f_up = (c + v) / 2Δ, f_down = (c − v) / 2Δ, and the difference f_up − f_down = v / Δ is the rate at which
nodes pass a fixed point. 

All positions are measured from the nozzle exit, and time is the model's t: t = 0 is a rising zero crossing of the
nozzle drive, h = h0 (1 + δ sin(2π f_nozzle t)).

Assumptions: plane waves along the axis, equal amplitudes, air at 20 °C, and rough attenuation of 1.6 dB/cm at 1 MHz
scaling with f². No transducer phase lag (calibrate that out).

In [7]:
import math

frequency = 17.5e3
dropRadius = 50e-6
flowRate = frequency*4/3*math.pi*(dropRadius)**3
nozzleRadius = 26.567e-6
jetVelocity = flowRate / (math.pi*nozzleRadius**2) # m/s
#print(f"flow rate: {flowRate}") # m^3 / s
# convert to pico liters
print(f"For a given nozzle radius {nozzleRadius*1e6} µm, with target drop radius of {dropRadius*1e6} µm...")
print(f"Flow Rate (V̇) = {round(flowRate * 1e6,4)} mL/s")  # 1 m^3 = 1e6 mL
print(f"Jet Velocity (v) = {round(jetVelocity,3)} m/s")   # 

# Target Standing wave
pressureAmplitude = 2000             # Pa, peak pressure at an antinode
dropletSpacing = 300e-6               # m
pressureNodesPerDropletSpacing = 1 
patternVelocity = 4.5                  # m/s, node velocity (= drop velocity)
nodePositionAtStart = 72.7e-6           # m, a node is here at t = 0

# Jet Excitations
nozzleFrequency = jetVelocity / (2 * math.sqrt(2) * math.pi * nozzleRadius)   # Hz, vjet / lambda_opt (main.cpp)
print(f"Nozzle f = {round(nozzleFrequency,2)} Hz") 

# Air Properties
airDensity = 1.204                      # kg/m^3
airSoundSpeed = 343.2                   # m/s
attenuationAt1MHz = 1.6                 # dB/cm, rough (depends on humidity). Adjust given target frequency.

# Geom
interactionPoint = 5e-3
upstreamTransducerPosition = 0.0        # annular transducer in the nozzle plane
downstreamTransducerPosition = 10e-3    # maybe something like 2x distance from IP?
transducerRadius = 5e-3                 # radiating face radius, for rough power estimates

For a given nozzle radius 26.567 µm, with target drop radius of 50.0 µm...
Flow Rate (V̇) = 0.0092 mL/s
Jet Velocity (v) = 4.132 m/s
Nozzle f = 17505.09 Hz


In [8]:
# Frequencies
nodeSpacing = dropletSpacing / pressureNodesPerDropletSpacing
frequencySum = airSoundSpeed / nodeSpacing
upstreamFrequency = 0.5 * (airSoundSpeed + patternVelocity) / nodeSpacing
downstreamFrequency = 0.5 * (airSoundSpeed - patternVelocity) / nodeSpacing
beatFrequency = upstreamFrequency - downstreamFrequency          # = patternVelocity / nodeSpacing

print(f"node spacing           {nodeSpacing * 1e6:10.2f} um")
print(f"upstream transducer    {upstreamFrequency:12.1f} Hz")
print(f"downstream transducer  {downstreamFrequency:12.1f} Hz")
print(f"difference             {beatFrequency:12.1f} Hz  (nozzle drive {nozzleFrequency:.1f} Hz)")
dropRate = patternVelocity / dropletSpacing
print(f"drops pass a point at  {dropRate:12.1f} Hz, {100 * (dropRate / nozzleFrequency - 1):+.2f}% from the nozzle frequency")

node spacing               300.00 um
upstream transducer        579500.0 Hz
downstream transducer      564500.0 Hz
difference                  15000.0 Hz  (nozzle drive 17505.1 Hz)
drops pass a point at       15000.0 Hz, -14.31% from the nozzle frequency


drops come off at the nozzle frequency, so set `f_up − f_down = pressureNodesPerDropletSpacing × f_nozzle` exactly (mix or synthesize both from the nozzle clock) and let the node spacing follow. A mismatch of x % slides the nodes off the drops by x % of a spacing every
drop period.

In [9]:
# Phases
# Drive signal, upstream cos(2π f_up t + phaseUp), downstream cos(2π f_down t + 0).
# With travel delays these give p_up = A cos(k_up z − ω_up t + φ_up) and p_down = A cos(k_down z + ω_down t + φ_down),
#   φ_up = −k_up z_up − phaseUp,   φ_down = −k_down z_down,
# and nodes where  k̄ z − π (f_up − f_down) t + (φ_up + φ_down)/2 = π/2 (mod π),  k̄ = (k_up + k_down)/2.
# A node at nodePositionAtStart when t = 0 needs  φ_up + φ_down = π − 2 k̄ z0  (mod 2π).
upstreamWavenumber = 2 * math.pi * upstreamFrequency / airSoundSpeed
downstreamWavenumber = 2 * math.pi * downstreamFrequency / airSoundSpeed
meanWavenumber = 0.5 * (upstreamWavenumber + downstreamWavenumber)

downstreamPhase = 0.0
phiDown = -downstreamWavenumber * downstreamTransducerPosition + downstreamPhase
phiUp = math.pi - 2 * meanWavenumber * nodePositionAtStart - phiDown
upstreamPhase = (-upstreamWavenumber * upstreamTransducerPosition - phiUp) % (2 * math.pi)

print(f"upstream drive phase    {math.degrees(upstreamPhase):7.2f} deg   (at t = 0)")
print(f"downstream drive phase  {math.degrees(downstreamPhase):7.2f} deg")

upstream drive phase     105.91 deg   (at t = 0)
downstream drive phase     0.00 deg


In [10]:
# Amplitude
# Each travelling wave carries half the antinode amplitude. Attenuation over the path from each face, different because different frequencies
def attenuationDecibels(frequency, distance):
    return attenuationAt1MHz * (frequency / 1e6) ** 2 * distance * 100

def soundPressureLevel(peakPressure):
    return 20 * math.log10(peakPressure / math.sqrt(2) / 20e-6)

waveAmplitudeAtInteraction = pressureAmplitude / 2
area = math.pi * transducerRadius**2
for name, frequency, position in [("upstream", upstreamFrequency, upstreamTransducerPosition),("downstream", downstreamFrequency, downstreamTransducerPosition)]:
    distance = abs(interactionPoint - position)
    loss = attenuationDecibels(frequency, distance)
    facePressure = waveAmplitudeAtInteraction * 10 ** (loss / 20)
    intensity = facePressure**2 / (2 * airDensity * airSoundSpeed)
    print(
    f"{name:10s}: {distance * 1e3:.1f} mm to the interaction point, {loss:.2f} dB loss -> "
    f"{facePressure:.0f} Pa peak at the face ({soundPressureLevel(facePressure):.1f} dB SPL), "
    f"{intensity / 1e4:.3f} W/cm^2, {intensity * area:.2f} W acoustic"
    )

upstream  : 5.0 mm to the interaction point, 0.27 dB loss -> 1031 Pa peak at the face (151.2 dB SPL), 0.129 W/cm^2, 0.10 W acoustic
downstream: 5.0 mm to the interaction point, 0.25 dB loss -> 1030 Pa peak at the face (151.2 dB SPL), 0.128 W/cm^2, 0.10 W acoustic


In [11]:
# checks
wavelength = airSoundSpeed / (0.5 * frequencySum)
nearField = transducerRadius**2 / wavelength
print(f"acoustic wavelength {wavelength * 1e6:.1f} um; near-field length a^2/λ = {nearField * 1e3:.0f} mm")

acoustic wavelength 600.0 um; near-field length a^2/λ = 42 mm


In [12]:
# rebuild
# signal of each wave at (z, t), including travel delay; the node is where |sum| is smallest.
def envelope(z, t):
    up = (2 * math.pi * upstreamFrequency * (t - (z - upstreamTransducerPosition) / airSoundSpeed) + upstreamPhase)
    down = (2 * math.pi * downstreamFrequency * (t - (downstreamTransducerPosition - z) / airSoundSpeed)
            + downstreamPhase)
    return abs(complex(math.cos(up), math.sin(up)) + complex(math.cos(down), math.sin(down)))

for t in [0.0, 10e-6, 25e-6]:
    expected = nodePositionAtStart + patternVelocity * t
    samples = [expected + (i - 500) * nodeSpacing / 1000 for i in range(1001)]  # ± half a spacing
    found = min(samples, key=lambda z: envelope(z, t))
    print(f"t = {t * 1e6:4.0f} us: node expected at {expected * 1e6:8.2f} um, found at {found * 1e6:8.2f} um")

t =    0 us: node expected at    72.70 um, found at    72.70 um
t =   10 us: node expected at   117.70 um, found at   117.70 um
t =   25 us: node expected at   185.20 um, found at   185.20 um
